# 🧾 Generador de Facturas Ficticias - Google Colab

Sistema automatizado para generar facturas peruanas con datos ficticios.

**Características:**
- ✅ 4 tipos de facturas: General, Hotel, Seguro, Con Descuento
- ✅ Exportación a PDF y JSON
- ✅ Guardado automático en Google Drive
- ✅ Cálculos validados (IGV 18%)
- ✅ Tipografía uniforme

---

## 📦 Paso 1: Instalación de Dependencias

In [ ]:
# Instalar reportlab para generar PDFs
!pip install reportlab python-dateutil faker -q

print("✅ Dependencias instaladas correctamente")

## 📂 Paso 2: Clonar Repositorio

In [ ]:
import os

# Clonar repositorio si no existe
if not os.path.exists('Creador-De-Factura'):
    !git clone https://github.com/GynoRomeroPrado/Creador-De-Factura.git
    print("✅ Repositorio clonado")
else:
    print("✅ Repositorio ya existe")

# Cambiar al directorio del proyecto
os.chdir('Creador-De-Factura')
print(f"📁 Directorio actual: {os.getcwd()}")

## 💾 Paso 3: Montar Google Drive

In [ ]:
from google.colab import drive
import os

# Montar Google Drive
drive.mount('/content/drive')

# Crear carpeta para facturas en Drive
DRIVE_PATH = '/content/drive/MyDrive/Facturas_Generadas'
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/PDFs", exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/JSONs", exist_ok=True)

print(f"✅ Google Drive montado")
print(f"📂 Facturas se guardarán en: {DRIVE_PATH}")

## ⚙️ Paso 4: Configuración

In [ ]:
# CONFIGURACIÓN - Ajusta estos valores según necesites

CANTIDAD_FACTURAS = 20  # Número de facturas a generar
GENERAR_PDF = True      # Generar PDFs
GENERAR_JSON = True     # Generar JSONs

print("📋 Configuración:")
print(f"   - Cantidad de facturas: {CANTIDAD_FACTURAS}")
print(f"   - Generar PDFs: {GENERAR_PDF}")
print(f"   - Generar JSONs: {GENERAR_JSON}")

## 🚀 Paso 5: Generar Facturas

In [ ]:
from src.generator import FacturaGenerator
from src.pdf_creator import PDFFactura
from src.json_exporter import JSONExporter
import shutil

print("="*70)
print("🧾 GENERADOR DE FACTURAS FICTICIAS")
print("="*70)

# Crear generadores
gen_factura = FacturaGenerator()
gen_pdf = PDFFactura(output_dir="temp_pdfs") if GENERAR_PDF else None
gen_json = JSONExporter(output_dir="temp_jsons") if GENERAR_JSON else None

facturas_generadas = []
archivos_pdf = []
archivos_json = []

# Tipos de facturas
tipos_factura = ['general', 'hotel', 'seguro', 'con_descuento']
categorias_items = ['construccion', 'comida', 'servicios', 'hoteles', 'combustibles', 'seguros', 'seguridad']

for i in range(CANTIDAD_FACTURAS):
    try:
        # Seleccionar tipo de factura de forma variada
        tipo = tipos_factura[i % len(tipos_factura)]
        
        # Ajustar categoría según tipo
        if tipo == 'hotel':
            categoria = 'hoteles'
        elif tipo == 'seguro':
            categoria = 'seguros'
        else:
            categoria = categorias_items[i % len(categorias_items)]
        
        # Variar moneda y crédito
        monedas_list = ['PEN', 'USD', 'EUR']
        moneda = monedas_list[i % len(monedas_list)]
        con_credito = (i % 3 == 0)
        
        # Generar factura
        factura = gen_factura.generar_factura(
            tipo_factura=tipo,
            categoria_items=categoria,
            num_items=None,
            moneda=moneda,
            con_credito=con_credito
        )
        
        facturas_generadas.append(factura)
        
        # Generar PDF
        if GENERAR_PDF:
            archivo_pdf = gen_pdf.crear_factura(factura)
            archivos_pdf.append(archivo_pdf)
        
        # Generar JSON
        if GENERAR_JSON:
            archivo_json = gen_json.exportar_factura(factura)
            archivos_json.append(archivo_json)
        
        # Mostrar progreso
        tipo_desc = f"[{tipo.upper()}]"
        print(f"[{i+1}/{CANTIDAD_FACTURAS}] ✓ {tipo_desc} {factura['numero_factura']} - {factura['simbolo_moneda']}{factura['total']:.2f}")
        
    except Exception as e:
        print(f"[{i+1}/{CANTIDAD_FACTURAS}] ✗ Error: {str(e)}")
        continue

print(f"\n{'='*70}")
print(f"✅ Generación completada: {len(facturas_generadas)}/{CANTIDAD_FACTURAS} facturas")
print(f"{'='*70}")

## 💾 Paso 6: Guardar en Google Drive

In [ ]:
import shutil
import os

print("💾 Copiando archivos a Google Drive...\n")

# Copiar PDFs
if GENERAR_PDF and archivos_pdf:
    print(f"📄 Copiando {len(archivos_pdf)} PDFs...")
    for archivo in archivos_pdf:
        destino = os.path.join(f"{DRIVE_PATH}/PDFs", os.path.basename(archivo))
        shutil.copy2(archivo, destino)
    print(f"   ✓ {len(archivos_pdf)} PDFs copiados a Drive/Facturas_Generadas/PDFs/")

# Copiar JSONs individuales
if GENERAR_JSON and archivos_json:
    print(f"\n📋 Copiando {len(archivos_json)} JSONs...")
    for archivo in archivos_json:
        destino = os.path.join(f"{DRIVE_PATH}/JSONs", os.path.basename(archivo))
        shutil.copy2(archivo, destino)
    print(f"   ✓ {len(archivos_json)} JSONs copiados a Drive/Facturas_Generadas/JSONs/")

# Crear archivo JSON consolidado
if GENERAR_JSON and facturas_generadas:
    print(f"\n📦 Creando archivo JSON consolidado...")
    archivo_consolidado = gen_json.exportar_multiple(facturas_generadas, "todas_facturas.json")
    destino_consolidado = os.path.join(f"{DRIVE_PATH}/JSONs", "todas_facturas.json")
    shutil.copy2(archivo_consolidado, destino_consolidado)
    print(f"   ✓ Archivo consolidado guardado")

print(f"\n{'='*70}")
print(f"✅ Todos los archivos guardados en Google Drive")
print(f"📂 Ubicación: {DRIVE_PATH}")
print(f"{'='*70}")

## 📊 Paso 7: Resumen y Estadísticas

In [ ]:
import json

if facturas_generadas and GENERAR_JSON:
    # Crear resumen
    resumen = gen_json.crear_resumen(facturas_generadas)
    
    print("="*70)
    print("📊 RESUMEN DE FACTURAS GENERADAS")
    print("="*70)
    print(f"\nTotal de facturas: {resumen['total_facturas']}")
    
    print(f"\n📋 Por tipo:")
    for tipo, cantidad in resumen['por_tipo'].items():
        print(f"   - {tipo.upper()}: {cantidad}")
    
    print(f"\n💰 Por moneda:")
    for moneda, cantidad in resumen['por_moneda'].items():
        print(f"   - {moneda}: {cantidad} facturas")
    
    print(f"\n💵 Totales por moneda:")
    for moneda, total in resumen['totales_por_moneda'].items():
        simbolo = {'PEN': 'S/', 'USD': '$', 'EUR': '€'}.get(moneda, '')
        print(f"   - {moneda}: {simbolo}{total:,.2f}")
    
    print(f"\n📈 Estadísticas:")
    print(f"   - Con crédito: {resumen['con_credito']} ({resumen['porcentaje_credito']}%)")
    print(f"   - Con descuento: {resumen['con_descuento']} ({resumen['porcentaje_descuento']}%)")
    
    # Guardar resumen en Drive
    resumen_path = os.path.join(f"{DRIVE_PATH}/JSONs", "resumen.json")
    with open(resumen_path, 'w', encoding='utf-8') as f:
        json.dump(resumen, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ Resumen guardado en: {resumen_path}")
    print("="*70)
else:
    print("⚠️ No hay facturas para resumir")

## ✅ Paso 8: Validar Cálculos (Opcional)

In [ ]:
print("🔍 Validando cálculos de facturas...\n")

errores_totales = 0

for i, factura in enumerate(facturas_generadas, 1):
    errores = []
    
    # Validar IGV
    igv_esperado = round(factura['op_gravada'] * 0.18, 2)
    if abs(igv_esperado - factura['igv']) > 0.02:
        errores.append(f"IGV incorrecto: esperado {igv_esperado}, obtenido {factura['igv']}")
    
    # Validar total
    total_esperado = round(
        factura['op_gravada'] + 
        factura['igv'] + 
        factura['op_exonerada'] + 
        factura['op_inafecta'] + 
        factura.get('total_cargos', 0) + 
        factura.get('otros_cargos', 0),
        2
    )
    
    if abs(total_esperado - factura['total']) > 0.02:
        errores.append(f"Total incorrecto: esperado {total_esperado}, obtenido {factura['total']}")
    
    # Validar cuotas
    if factura['con_credito'] and factura['cuotas']:
        suma_cuotas = sum(cuota['monto'] for cuota in factura['cuotas'])
        if abs(suma_cuotas - factura['total']) > 0.02:
            errores.append(f"Suma de cuotas no coincide: {suma_cuotas} vs {factura['total']}")
    
    if errores:
        print(f"❌ Factura {i} ({factura['numero_factura']}):")
        for error in errores:
            print(f"   - {error}")
        errores_totales += len(errores)

if errores_totales == 0:
    print("✅ TODAS LAS VALIDACIONES PASARON CORRECTAMENTE!")
    print(f"   {len(facturas_generadas)} facturas validadas sin errores")
else:
    print(f"\n⚠️ Se encontraron {errores_totales} errores en las validaciones")

## 📥 Paso 9: Descargar Archivos (Opcional)

Si quieres descargar los archivos directamente a tu computadora:

In [ ]:
from google.colab import files
import zipfile

# Crear ZIP con todos los archivos
zip_path = "/content/facturas_generadas.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Agregar PDFs
    if GENERAR_PDF:
        for archivo in archivos_pdf:
            zipf.write(archivo, f"PDFs/{os.path.basename(archivo)}")
    
    # Agregar JSONs
    if GENERAR_JSON:
        for archivo in archivos_json:
            zipf.write(archivo, f"JSONs/{os.path.basename(archivo)}")
        
        # Agregar consolidado y resumen
        if os.path.exists("temp_jsons/todas_facturas.json"):
            zipf.write("temp_jsons/todas_facturas.json", "JSONs/todas_facturas.json")

print(f"📦 Archivo ZIP creado: {zip_path}")
print(f"📥 Descargando...")

# Descargar
files.download(zip_path)

print("✅ Descarga completada")

---

## 🎉 ¡Listo!

Tus facturas han sido generadas y guardadas en:
- **Google Drive:** `MyDrive/Facturas_Generadas/`
  - PDFs en `PDFs/`
  - JSONs en `JSONs/`
  - Archivo consolidado: `JSONs/todas_facturas.json`
  - Resumen: `JSONs/resumen.json`

### 📝 Notas:
- Todas las facturas tienen cálculos validados (IGV 18%)
- Los RUCs son válidos en formato pero ficticios
- Las fechas son del año 2025
- Tipografía uniforme Helvetica en PDFs

### 🔄 Para generar más facturas:
Simplemente cambia el valor de `CANTIDAD_FACTURAS` en el Paso 4 y vuelve a ejecutar desde ahí.

---

**Desarrollado con ❤️ para generar facturas ficticias peruanas**